# Cyberbullying Classification in Tweets

This notebook documents a compact NLP workflow for comparing lexical and semantic text representations in multiclass cyberbullying classification.


## 1. Imports


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
from IPython.display import display
import pandas as pd
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split


## 2. Load dataset


In [ ]:
DATA_DIR = Path('data')
dataset_candidates = sorted(DATA_DIR.glob('*.csv'))
if not dataset_candidates:
    raise FileNotFoundError('Add the dataset CSV to the data/ directory before running the notebook.')

df = pd.read_csv(dataset_candidates[0])
expected_columns = {'tweet', 'type'}
missing_columns = expected_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Missing required columns: {sorted(missing_columns)}')

df = df[['tweet', 'type']].dropna().copy()
df.head()


## 3. Data exploration


In [ ]:
print(df.shape)
display(df['type'].value_counts().sort_index())

plt.figure(figsize=(8, 4))
sns.countplot(data=df, y='type', order=sorted(df['type'].unique()))
plt.title('Class distribution')
plt.tight_layout()
plt.savefig('images/class_distribution.png', dpi=200, bbox_inches='tight')
plt.show()


## 4. Data preparation


In [ ]:
def clean_tweet(text: str) -> str:
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_tweet'] = df['tweet'].astype(str).map(clean_tweet)
df[['tweet', 'clean_tweet', 'type']].head()


## 5. Train/test split


In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

X = df['clean_tweet']
y = df['type']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)


## 6. TF-IDF baseline


In [ ]:
def evaluate_predictions(y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    macro = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    weighted = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_precision': macro[0],
        'macro_recall': macro[1],
        'macro_f1': macro[2],
        'weighted_precision': weighted[0],
        'weighted_recall': weighted[1],
        'weighted_f1': weighted[2],
        'per_class_f1': {label: metrics['f1-score'] for label, metrics in report.items() if isinstance(metrics, dict)},
    }

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
tfidf_model = LogisticRegression(max_iter=1000)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
tfidf_model.fit(X_train_tfidf, y_train)
tfidf_predictions = tfidf_model.predict(X_test_tfidf)
tfidf_results = evaluate_predictions(y_test, tfidf_predictions)
pd.Series({k: v for k, v in tfidf_results.items() if k != 'per_class_f1'})


## 7. Sentence Transformer experiment


In [ ]:
# Replace the placeholder below with the exact sentence-transformer model used in the experiment.
SENTENCE_MODEL_NAME = 'sentence-transformer-model-used-in-your-experiment'
sentence_model = SentenceTransformer(SENTENCE_MODEL_NAME)

X_train_embeddings = sentence_model.encode(X_train.tolist(), show_progress_bar=True)
X_test_embeddings = sentence_model.encode(X_test.tolist(), show_progress_bar=True)

embedding_model = LogisticRegression(max_iter=1000)
embedding_model.fit(X_train_embeddings, y_train)
embedding_predictions = embedding_model.predict(X_test_embeddings)
embedding_results = evaluate_predictions(y_test, embedding_predictions)
pd.Series({k: v for k, v in embedding_results.items() if k != 'per_class_f1'})


## 8. Model comparison


In [ ]:
comparison_df = pd.DataFrame([
    {'representation': 'TF-IDF', 'accuracy': tfidf_results['accuracy'], 'macro_f1': tfidf_results['macro_f1']},
    {'representation': 'Sentence Transformer embeddings', 'accuracy': embedding_results['accuracy'], 'macro_f1': embedding_results['macro_f1']},
])
comparison_df

ax = comparison_df.set_index('representation')[['accuracy', 'macro_f1']].plot(kind='bar', figsize=(8, 4))
ax.set_ylabel('Score')
ax.set_title('Model comparison')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('images/model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## 9. Five-class experiment


In [ ]:
five_class_df = df[df['type'] != 'other_cyberbullying'].copy()

X_five = five_class_df['clean_tweet']
y_five = five_class_df['type']

X_train_5, X_test_5, y_train_5, y_test_5 = train_test_split(
    X_five,
    y_five,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_five,
)

tfidf_vectorizer_5 = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
tfidf_model_5 = LogisticRegression(max_iter=1000)

X_train_tfidf_5 = tfidf_vectorizer_5.fit_transform(X_train_5)
X_test_tfidf_5 = tfidf_vectorizer_5.transform(X_test_5)
tfidf_model_5.fit(X_train_tfidf_5, y_train_5)
tfidf_predictions_5 = tfidf_model_5.predict(X_test_tfidf_5)
five_class_results = evaluate_predictions(y_test_5, tfidf_predictions_5)
pd.Series({k: v for k, v in five_class_results.items() if k != 'per_class_f1'})


## 10. Confusion matrix


In [ ]:
five_class_labels = sorted(y_train_5.unique())
cm = confusion_matrix(y_test_5, tfidf_predictions_5, labels=five_class_labels)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=five_class_labels, yticklabels=five_class_labels)
plt.title('Five-class confusion matrix')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
plt.savefig('images/confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.show()


## 11. Error analysis


In [ ]:
error_analysis_df = pd.DataFrame({
    'tweet': X_test_5.values,
    'true_label': y_test_5.values,
    'predicted_label': tfidf_predictions_5,
})

gender_to_non = error_analysis_df[(error_analysis_df['true_label'] == 'gender') & (error_analysis_df['predicted_label'] == 'not_cyberbullying')]
non_to_gender = error_analysis_df[(error_analysis_df['true_label'] == 'not_cyberbullying') & (error_analysis_df['predicted_label'] == 'gender')]

print('gender -> not_cyberbullying:', len(gender_to_non))
print('not_cyberbullying -> gender:', len(non_to_gender))

display(gender_to_non.head(10))
display(non_to_gender.head(10))


## 12. Conclusions

Reported project results:

- **Six-class TF-IDF + Logistic Regression** — Accuracy: `0.8177088863`, Macro F1: `0.8187764435`
- **Six-class Sentence Transformer + Logistic Regression** — Accuracy: `0.8105956046`, Macro F1: `0.8084662255`
- **Five-class TF-IDF + Logistic Regression** — Accuracy: `0.9277032160`, Macro F1: `0.9282693468`

The five-class result should be interpreted as a separate experiment with a different target space, not as a direct like-for-like improvement over the six-class setup.
